In [ ]:
import os, zipfile, hashlib
from pathlib import Path

DRIVE_ZIP_PATH = "/content/drive/MyDrive/beefcattle_muzzle.zip"
EXTRACT_DIR = "/content/beefcattle_muzzle"

from google.colab import drive
drive.mount("/content/drive")

In [6]:
def sha256_of_file(path, chunk_size=8192):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()

dataset_zip_hash = sha256_of_file(DRIVE_ZIP_PATH)
print(dataset_zip_hash)

1b8815e76099569b968a35b8333e8b76f39341c4fcd9993d4b16e37fb9a00777


In [7]:
os.makedirs(EXTRACT_DIR, exist_ok=True)

with zipfile.ZipFile(DRIVE_ZIP_PATH, "r") as zf:
    zf.extractall(EXTRACT_DIR)

# Manifest

In [9]:
import pandas as pd

root = Path(EXTRACT_DIR) / "beefcattle_muzzle"
records = []

for cattle_dir in sorted(root.iterdir()):
    if not cattle_dir.is_dir():
        continue
    cattle_id = cattle_dir.name
    images = [p for p in cattle_dir.iterdir() if p.suffix.lower() in (".jpg", ".jpeg", ".png")]
    for img_path in images:
        records.append({"image_path": str(img_path), "cattle_id": cattle_id})

manifest = pd.DataFrame(records)
manifest.to_csv("dataset_manifest.csv", index=False)

print(f"folders found: {sum(1 for d in root.iterdir() if d.is_dir())}")
print(f"cattle with >=1 image: {manifest['cattle_id'].nunique()}")
print(f"total images: {len(manifest)}")
print(manifest.groupby("cattle_id").size().describe())

folders found: 268
cattle with >=1 image: 268
total images: 4923
count    268.000000
mean      18.369403
std       10.165380
min        4.000000
25%       11.750000
50%       16.000000
75%       25.000000
max       70.000000
dtype: float64
